## 🔍 Stop Loss Logic Verification Summary

### ✅ Correct Implementation (Both cells now fixed)

**Key components of proper stop loss logic:**

1. **Trading Costs Applied:**
   - Entry: `entry_price = price * (1 + trading_cost_pct)` 
   - Exit: `exit_price = price * (1 - trading_cost_pct)`

2. **Stop Loss Priority (checked FIRST):**
   ```python
   if price <= stop_loss_price:  # Priority 1: Stop loss
       exit and calculate return
   elif prob <= sell_threshold:  # Priority 2: Sell signal
       exit and calculate return
   ```

3. **Percentage-Based Returns:**
   - ✅ `balance *= (exit_price / entry_price)` - Correct
   - ❌ `balance += (sell_price - buy_price)` - WRONG (was in old code)

4. **Stop Loss Calculation:**
   - `stop_loss_price = entry_price * (1 - stop_loss_pct)`
   - For 2% stop loss: exits when price drops 2% below entry price

### 📊 Both Cells Are Now Consistent:
- **Cell 12**: Simple strategy execution with detailed trade history
- **Cell 13**: Threshold optimization with stop loss (grid search)

## load model and test it and backtest based on balance and risk management techniques 


In [1]:
import os
import json
import pickle
import numpy as np
import pandas as pd
import yfinance as yf
import talib as tal
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

# Try to import TensorFlow/Keras for neural network models
try:
    import tensorflow as tf
    from tensorflow import keras
    
    KERAS_AVAILABLE = True
    print("✅ TensorFlow/Keras available")
except ImportError:
    KERAS_AVAILABLE = False
    print("⚠️ TensorFlow/Keras not available - only sklearn models supported")

print("✅ Libraries imported successfully")

✅ TensorFlow/Keras available
✅ Libraries imported successfully


In [2]:
# ============================================================================
# LOAD ALL 1m MODELS FROM RESULTS FOLDER
# ============================================================================
import os
import json
import pickle
import pandas as pd

# Path to the results root folder
RESULTS_ROOT = r"C:\Users\aram\OneDrive\Bureau\ding\testing a strategy\python_classes\results"

def load_all_1m_models():
    """Load all models from the 1m results folder."""
    models_1m = {}

    # Find the 1m results folder
    results_1m_path = None
    for folder_name in os.listdir(RESULTS_ROOT):
        if folder_name.startswith('ETH_USD_1m_') and os.path.isdir(os.path.join(RESULTS_ROOT, folder_name)):
            results_1m_path = os.path.join(RESULTS_ROOT, folder_name)
            break

    if results_1m_path is None:
        print("❌ No 1m results folder found!")
        return models_1m

    print(f"📂 Found 1m results folder: {os.path.basename(results_1m_path)}")

    # Load config and results
    config_path = os.path.join(results_1m_path, "config.json")
    results_path = os.path.join(results_1m_path, "results_summary.json")
    registry_path = os.path.join(results_1m_path, "models_registry.json")
    features_path = os.path.join(results_1m_path, "features.json")

    if not os.path.exists(config_path) or not os.path.exists(results_path):
        print("❌ Missing config or results files!")
        return models_1m

    # Load metadata
    config = json.load(open(config_path))

    results = json.load(open(results_path))

    models_registry = None
    if os.path.exists(registry_path):
        try:
            models_registry = json.load(open(registry_path))
        except:
            models_registry = None

    features_config = None
    if os.path.exists(features_path):
        try:
            features_config = json.load(open(features_path))
            
            # Set global feature lists for later use (mapping correctly to json keys)
            # Check multiple potential keys for robustness
            full_keys = [ 'X_full','X_full_features']
            sel_keys = ['X_selected','X_features']
            
            for k in full_keys:
                if k in features_config:
                    globals()['X_FULL_FEATURES_1m'] = features_config[k]
                    print(f"✅ Loaded X_FULL_FEATURES_1m ({len(features_config[k])} features)")
                    break
                    
            for k in sel_keys:
                if k in features_config:
                    globals()['X_FEATURES_1m'] = features_config[k]
                    print(f"✅ Loaded X_FEATURES_1m ({len(features_config[k])} features)")
                    break
        except Exception as e:
            print(f"⚠️ Failed to load features info: {e}")
            features_config = None

    # Get all models from ranking
    ranking = results.get('ranking', [])
    all_models_info = results.get('all_models', {})

    print(f"🔍 Found {len(ranking)} models in 1m results")
    print(f"🏆 Best model: {results.get('best_model', 'N/A')}")

    # Load each model
    models_dir = os.path.join(results_1m_path, "models")
    loaded_count = 0

    for model_info in ranking:
        model_name = model_info['model']

        # Find model file
        model_path = None
        model_type = None

        # Check models directory first
        if os.path.exists(models_dir):
            for ext in ['.keras', '.pkl']:
                candidate_path = os.path.join(models_dir, f"{model_name}{ext}")
                if os.path.exists(candidate_path):
                    model_path = candidate_path
                    model_type = 'keras' if ext == '.keras' else 'pkl'
                    break

        # Check root directory if not found
        if model_path is None:
            for ext in ['.keras', '.pkl']:
                candidate_path = os.path.join(results_1m_path, f"{model_name}{ext}")
                if os.path.exists(candidate_path):
                    model_path = candidate_path
                    model_type = 'keras' if ext == '.keras' else 'pkl'
                    break

        if model_path is None:
            print(f"⚠️ Model file not found for: {model_name}")
            continue

        # Load the model
        try:
            if model_type == 'keras' and KERAS_AVAILABLE:
                model = keras.models.load_model(model_path)
            elif model_type in ['pkl', 'sklearn']:
                with open(model_path, 'rb') as f:
                    model = pickle.load(f)
            else:
                print(f"⚠️ Unsupported model type {model_type} for: {model_name}")
                continue

            # Get additional info
            model_class = None
            if models_registry:
                model_entry = models_registry.get('models', {}).get(model_name, {})
                model_class = model_entry.get('model_class')

            # Store model info
            models_1m[model_name] = {
                'model': model,
                'model_type': model_type,
                'model_class': model_class,
                'model_path': model_path,
                'feature_set': model_info.get('feature_set', 'X'),
                'train_acc': model_info.get('train_acc', 0),
                'forward_acc': model_info.get('forward_acc', 0),
                'rank': model_info.get('rank', 999),
                'config': config,
                'features_config': features_config
            }

            loaded_count += 1
            print(f"✅ Loaded {model_name} ({model_type}) - Rank: {model_info.get('rank', 'N/A')}, Acc: {model_info.get('forward_acc', 0):.4f}")

        except Exception as e:
            print(f"❌ Failed to load {model_name}: {e}")
            continue

    print(f"\n📊 Successfully loaded {loaded_count} out of {len(ranking)} 1m models")
    return models_1m

# ============================================================================
# EXECUTE LOADING
# ============================================================================
print("=" * 70)
print("🚀 LOADING ALL 1m MODELS")
print("=" * 70)

MODELS_1m = load_all_1m_models()

if len(MODELS_1m) == 0:
    print("❌ No 1m models were loaded!")
else:
    print(f"\n📈 Available 1m models: {list(MODELS_1m.keys())}")

    # Show summary
    df_1m_models = pd.DataFrame([
        {
            'Model': name,
            'Type': info['model_type'],
            'Class': info['model_class'] or 'N/A',
            'Rank': info['rank'],
            'Train_Acc': f"{info['train_acc']:.4f}",
            'Forward_Acc': f"{info['forward_acc']:.4f}",
            'Feature_Set': info['feature_set']
        }
        for name, info in MODELS_1m.items()
    ]).sort_values('Rank')

    print("\n🏆 1m MODELS SUMMARY (sorted by rank):")
    display(df_1m_models)

print("=" * 70)

🚀 LOADING ALL 1m MODELS
📂 Found 1m results folder: ETH_USD_1m_20260224_091547
✅ Loaded X_FULL_FEATURES_1m (28 features)
✅ Loaded X_FEATURES_1m (18 features)
🔍 Found 28 models in 1m results
🏆 Best model: CatBoost_X
✅ Loaded CatBoost_X (pkl) - Rank: 1, Acc: 0.5226
✅ Loaded CatBoost_Full (pkl) - Rank: 2, Acc: 0.5213
✅ Loaded LightGBM_X (pkl) - Rank: 3, Acc: 0.5174
✅ Loaded XGBoost_X (pkl) - Rank: 4, Acc: 0.5157
✅ Loaded GRU_Full (keras) - Rank: 5, Acc: 0.5157
✅ Loaded LSTM_X (keras) - Rank: 6, Acc: 0.5148
✅ Loaded LightGBM_Full (pkl) - Rank: 7, Acc: 0.5143
✅ Loaded AdaBoost_X (pkl) - Rank: 8, Acc: 0.5142
✅ Loaded RandomForest_X (pkl) - Rank: 9, Acc: 0.5137
✅ Loaded AdaBoost_Full (pkl) - Rank: 10, Acc: 0.5137
✅ Loaded RandomForest_Full (pkl) - Rank: 11, Acc: 0.5130
✅ Loaded XGBoost_Full (pkl) - Rank: 12, Acc: 0.5130
✅ Loaded LSTM_Attention_X (keras) - Rank: 13, Acc: 0.5119
✅ Loaded LSTM_Full (keras) - Rank: 14, Acc: 0.5116
✅ Loaded LSTM_Attention_Full (keras) - Rank: 15, Acc: 0.5113
✅ Load

,Model,Type,Class,Rank,Train_Acc,Forward_Acc,Feature_Set
0,CatBoost_X,pkl,CatBoostClassifier,1,0.5217,0.5226,X
1,CatBoost_Full,pkl,CatBoostClassifier,2,0.5196,0.5213,Full
2,LightGBM_X,pkl,LGBMClassifier,3,0.5200,0.5174,X
3,XGBoost_X,pkl,XGBClassifier,4,0.5202,0.5157,X
4,GRU_Full,keras,Sequential,5,0.5170,0.5157,Full
5,LSTM_X,keras,Sequential,6,0.5194,0.5148,X
6,LightGBM_Full,pkl,LGBMClassifier,7,0.5218,0.5143,Full
7,AdaBoost_X,pkl,AdaBoostClassifier,8,0.5200,0.5142,X
8,RandomForest_X,pkl,RandomForestClassifier,9,0.5201,0.5137,X
9,AdaBoost_Full,pkl,AdaBoostClassifier,10,0.5196,0.5137,Full


In [3]:
# Load scaler values from config
RESULTS_ROOT = r"C:\Users\aram\OneDrive\Bureau\ding\testing a strategy\python_classes\results"
RESULTS_1m_FOLDER = None
for folder_name in os.listdir(RESULTS_ROOT):
    if folder_name.startswith('ETH_USD_1m_') and os.path.isdir(os.path.join(RESULTS_ROOT, folder_name)):
        RESULTS_1m_FOLDER = os.path.join(RESULTS_ROOT, folder_name)
        break

if RESULTS_1m_FOLDER:
    with open(os.path.join(RESULTS_1m_FOLDER, "config.json"), "r") as f:
        config_1m = json.load(f)
    
    scaler_1m = config_1m["scaler"]
    print(f"Config loaded from: {os.path.basename(RESULTS_1m_FOLDER)}")
    print(f"Scaler keys: {list(scaler_1m.keys())}")
    
    x_mean_dict_1m = scaler_1m["X_mean"]
    x_std_dict_1m = scaler_1m["X_std"]
    x_full_mean_dict_1m = scaler_1m["X_full_mean"]
    x_full_std_dict_1m = scaler_1m["X_full_std"]
else:
    print("ERROR: No 1m results folder found!")

Config loaded from: ETH_USD_1m_20260224_091547
Scaler keys: ['X_mean', 'X_std', 'X_full_mean', 'X_full_std']


In [5]:
# ============================================================================
# LOAD 1M DATA FROM YFINANCE (ADDITIONAL CELL)
# ============================================================================
import yfinance as yf
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

SYMBOL_1M_EXTRA = 'ETH-USD'  # Change if needed

print("🚀 Fetching 1m data (additional cell) for:", SYMBOL_1M_EXTRA)

try:
    ticker_extra = yf.Ticker(SYMBOL_1M_EXTRA)

    # Primary: max period
    df_1m_extra = ticker_extra.history(period='max', interval='1m', prepost=True, actions=False)

    # Fallbacks if empty
    if df_1m_extra is None or df_1m_extra.empty:
        print("⚠️ primary attempt empty — trying period='730d'...")
        df_1m_extra = ticker_extra.history(period='7d', interval='1m', prepost=True, actions=False)

    if df_1m_extra is None or df_1m_extra.empty:
        print("⚠️ still empty — trying yf.download(period='60d')...")
        df_1m_extra = yf.download(SYMBOL_1M_EXTRA, period='7d', interval='1m', progress=False)

    if df_1m_extra is None or df_1m_extra.empty:
        print("❌ No 1m data retrieved from yfinance in this cell.")
        DF_1M_MAX_EXTRA = pd.DataFrame()
    else:
        # Clean
        df_1m_extra = df_1m_extra[~df_1m_extra.index.duplicated(keep='first')]
        df_1m_extra = df_1m_extra.sort_index()
        df_1m_extra = df_1m_extra.dropna(how='all')
        df_1m_extra = df_1m_extra.ffill()        # Derived columns
        df_1m_extra['Returns'] = df_1m_extra['Close'].pct_change()
        df_1m_extra['Log_Return'] = np.log(df_1m_extra['Close'] / df_1m_extra['Close'].shift(1))

        # Store in a separate variable to avoid overwriting
        DF_1M_MAX_EXTRA = df_1m_extra.copy()

        print("✅ Data stored in variable: DF_1M_MAX_EXTRA")
        print(f"   Shape: {DF_1M_MAX_EXTRA.shape}")
        print(f"   Date range: {DF_1M_MAX_EXTRA.index[0]} to {DF_1M_MAX_EXTRA.index[-1]}")

        print("\n🔍 FIRST 3 ROWS:")
        display(DF_1M_MAX_EXTRA.head(3))
        print("\n🔍 LAST 3 ROWS:")
        display(DF_1M_MAX_EXTRA.tail(3))

except Exception as e:
    print(f"❌ Error fetching data: {e}")
    import traceback
    traceback.print_exc()
    DF_1M_MAX_EXTRA = pd.DataFrame()


🚀 Fetching 1m data (additional cell) for: ETH-USD
✅ Data stored in variable: DF_1M_MAX_EXTRA
   Shape: (8411, 7)
   Date range: 2026-02-18 07:07:00+00:00 to 2026-02-26 07:04:00+00:00

🔍 FIRST 3 ROWS:


,Open,High,Low,Close,Volume,Returns,Log_Return
Datetime,,,,,,,
2026-02-18 07:07:00+00:00,2001.832031,2001.832031,2001.832031,2001.832031,0,NaN,NaN
2026-02-18 07:08:00+00:00,2002.457153,2002.457153,2002.457153,2002.457153,0,0.000312,0.000312
2026-02-18 07:10:00+00:00,2002.446533,2002.446533,2002.446533,2002.446533,138745856,-0.000005,-0.000005



🔍 LAST 3 ROWS:


,Open,High,Low,Close,Volume,Returns,Log_Return
Datetime,,,,,,,
2026-02-26 07:02:00+00:00,2065.353516,2065.353516,2065.353516,2065.353516,41080832,0.000719,0.000719
2026-02-26 07:03:00+00:00,2064.288330,2064.288330,2064.288330,2064.288330,3725312,-0.000516,-0.000516
2026-02-26 07:04:00+00:00,2064.850098,2064.850098,2064.850098,2064.850098,0,0.000272,0.000272


In [6]:
import yfinance as yf

def calculate_indicators(df):
    """
    Calculate technical indicators for 1m feature set only.
    """
    df = df.copy()

    # Target variable (for reference)
    df["return"] = df["Close"].pct_change()
    df["target"] = (df["return"].shift(-1) > 0).astype(int)

    # Drop only rows where OHLCV are all NaN
    df = df.dropna(subset=['Open', 'High', 'Low', 'Close', 'Volume'], how='all')
    
    # Forward fill missing OHLCV values
    df[['Open', 'High', 'Low', 'Close', 'Volume']] = df[['Open', 'High', 'Low', 'Close', 'Volume']].ffill()
    # -----------------------------
    # HTF (Higher Timeframe) features (from yfinance)
    # Handle gracefully if HTF calculation fails
    # -----------------------------
    try:
        
        # Get the date range from the 1m dataframe
        start_date = df.index.min()
        end_date = df.index.max()
        
        # Download 5m and 15m data from yfinance
        ticker = yf.Ticker(SYMBOL_1M_EXTRA)
        
        for label, interval in [("5m", "5m"), ("15m", "15m")]:
            # Download data for the specific interval
            htf_data = ticker.history(start=start_date, end=end_date, interval=interval)
            
            if htf_data.empty:
                raise ValueError(f"No data downloaded for {label} interval")
            
            # Calculate features
            res_close = htf_data["Close"]
            res_ret = res_close.pct_change()
            res_log = np.log(res_close / res_close.shift(1))
            
            # Create HTF dataframe with shifted values
            htf_df = pd.DataFrame({
                f"HTF_{label}_Close_prev": res_close.shift(1),
                f"HTF_{label}_Return_prev": res_ret.shift(1),
                f"HTF_{label}_LogReturn_prev": res_log.shift(1),
            })
            
            # Reindex to match 1m dataframe and forward fill
            htf_intraday = htf_df.reindex(df.index).ffill()
            
            # Join with main dataframe
            df = df.join(htf_intraday)
            
    except Exception as e:
        print(f"⚠️ HTF 5m/15m feature calculation failed: {e}")
        for col in [
            "HTF_5m_Close_prev", "HTF_5m_Return_prev", "HTF_5m_LogReturn_prev",
            "HTF_15m_Close_prev", "HTF_15m_Return_prev", "HTF_15m_LogReturn_prev",
        ]:
            df[col] = 0.0  # Fill with zeros instead of NaN

    # Trend Indicators
    df["EMA_20"] = tal.EMA(df["Close"].values, timeperiod=20)
    df["SAR"] = tal.SAR(df['High'].values, df['Low'].values, acceleration=0.02, maximum=0.2)
    df["SMA_10"] = df["Close"].rolling(10).mean()
    df["SMA_20"] = df["Close"].rolling(20).mean()
    df["SMA_50"] = df["Close"].rolling(50).mean()
    df["STD_10"] = df["Close"].rolling(10).std()
    df["STD_20"] = df["Close"].rolling(20).std()

    # Momentum Indicators
    df["RSI_20"] = tal.RSI(df["Close"].values, timeperiod=20)
    df["ROC_10"] = df["Close"].pct_change(10)
    df["CCI_20"] = tal.CCI(df['High'].values, df['Low'].values, df['Close'].values, timeperiod=20)
    df["MACD"], df["MACD_signal"], df["MACD_hist"] = tal.MACD(df['Close'].values)

    # Volatility Indicators
    df["ATR_14"] = tal.ATR(df['High'].values, df['Low'].values, df['Close'].values, timeperiod=14)
    sma = df["Close"].rolling(20).mean()
    std = df["Close"].rolling(20).std()
    df["BB_upper"] = sma + (std * 2)
    df["BB_lower"] = sma - (std * 2)
    df["BW_20"] = (df["BB_upper"] - df["BB_lower"]) / sma

    # Volume lags
    df["Volume_lag1"] = df["Volume"].shift(1)
    df["Volume_lag2"] = df["Volume"].shift(2)

    # Chaikin Money Flow (CMF)
    mfm = ((df['Close'] - df['Low']) - (df['High'] - df['Close'])) / (df['High'] - df['Low'])
    mfm = mfm.fillna(0)  # Handle division by zero when High == Low
    mfv = mfm * df['Volume']
    df["CMF_20"] = mfv.rolling(20).sum() / df['Volume'].rolling(20).sum()
    df["CMF_20"] = df["CMF_20"].fillna(0)  # Fill NaN with 0

    # Price-based Indicators
    df["Log_Return"] = np.log(df["Close"] / df["Close"].shift(1))

    # Lag features
    for i in range(1, 6):
        df[f"Lag_{i}"] = df["return"].shift(i)

    # Keep only required 1m feature columns + target/return
    required_cols = [
        "Open",
        "Close",
        "Volume",
        "HTF_5m_Close_prev",
        "HTF_5m_Return_prev",
        "HTF_5m_LogReturn_prev",
        "HTF_15m_Close_prev",
        "HTF_15m_Return_prev",
        "HTF_15m_LogReturn_prev",
        "EMA_20",
        "SAR",
        "RSI_20",
        "ROC_10",
        "CCI_20",
        "MACD",
        "MACD_signal",
        "MACD_hist",
        "ATR_14",
        "BB_lower",
        "BW_20",
        "SMA_10",
        "SMA_20",
        "SMA_50",
        "STD_10",
        "STD_20",
        "CMF_20",
        "Log_Return",
        "Lag_1",
        "Lag_2",
        "Lag_3",
        "Lag_4",
        "Lag_5",
        "Volume_lag1",
        "Volume_lag2",
    ]

    # Forward fill remaining NaNs from indicator calculations
    df[required_cols] = df[required_cols].ffill().bfill()
    
    # Drop rows with NaNs only in required columns (final cleanup)
    df = df.dropna(subset=required_cols + ["target", "return"])

    return df[required_cols + ["target", "return"]]

In [7]:
# ============================================================================
# PROCESS RAW DATA TO CREATE df_used
# ============================================================================

# Process the loaded 1m data with indicators
if 'DF_1M_MAX_EXTRA' in globals() and isinstance(DF_1M_MAX_EXTRA, pd.DataFrame) and not DF_1M_MAX_EXTRA.empty:
    print("🔄 Processing raw data with indicators...")
    df_used = calculate_indicators(DF_1M_MAX_EXTRA)
    print(f"✅ df_used created successfully")
    print(f"   Shape: {df_used.shape}")
    print(f"   Columns: {list(df_used.columns)}")
    print(f"   Date range: {df_used.index[0]} to {df_used.index[-1]}")
elif 'DF_1M_MAX' in globals() and isinstance(DF_1M_MAX, pd.DataFrame) and not DF_1M_MAX.empty:
    print("🔄 Processing DF_1M_MAX with indicators...")
    df_used = calculate_indicators(DF_1M_MAX)
    print(f"✅ df_used created successfully")
    print(f"   Shape: {df_used.shape}")
else:
    print("❌ No raw 1m data available (DF_1M_MAX_EXTRA or DF_1M_MAX). Please run the data loading cell first.")

🔄 Processing raw data with indicators...
✅ df_used created successfully
   Shape: (8410, 36)
   Columns: ['Open', 'Close', 'Volume', 'HTF_5m_Close_prev', 'HTF_5m_Return_prev', 'HTF_5m_LogReturn_prev', 'HTF_15m_Close_prev', 'HTF_15m_Return_prev', 'HTF_15m_LogReturn_prev', 'EMA_20', 'SAR', 'RSI_20', 'ROC_10', 'CCI_20', 'MACD', 'MACD_signal', 'MACD_hist', 'ATR_14', 'BB_lower', 'BW_20', 'SMA_10', 'SMA_20', 'SMA_50', 'STD_10', 'STD_20', 'CMF_20', 'Log_Return', 'Lag_1', 'Lag_2', 'Lag_3', 'Lag_4', 'Lag_5', 'Volume_lag1', 'Volume_lag2', 'target', 'return']
   Date range: 2026-02-18 07:08:00+00:00 to 2026-02-26 07:04:00+00:00


In [23]:
# ============================================================================
# Sync feature lists with expected definitions and realign dataframes
# ============================================================================

# Expected feature definitions
expected_x_features = [
    "Open", "Volume", "RSI_20", "ROC_10", "CCI_20", "MACD", "MACD_hist",
    "ATR_14", "BW_20", "CMF_20", "Log_Return",
    "Lag_1", "Lag_2", "Lag_3", "Lag_4", "Lag_5", "Volume_lag1", "Volume_lag2",
]
expected_x_full_features = [
    "Open", "Close", "Volume", "EMA_20", "SAR", "RSI_20", "ROC_10", "CCI_20",
    "MACD", "MACD_signal", "MACD_hist", "ATR_14", "BB_lower", "BW_20",
    "SMA_10", "SMA_20", "SMA_50", "STD_10", "STD_20", "CMF_20", "Log_Return",
    "Lag_1", "Lag_2", "Lag_3", "Lag_4", "Lag_5", "Volume_lag1", "Volume_lag2",
]

# Update global feature lists
X_FEATURES_1m = expected_x_features
X_FULL_FEATURES_1m = expected_x_full_features

# Update features_config if present
if "features_config" in globals() and isinstance(features_config, dict):
    features_config["X_features"] = expected_x_features
    features_config["X_full_features"] = expected_x_full_features
    features_config["best_model_feature_set"] = "X"

# Helper to ensure columns exist (pad missing with zeros, preserve order)
if "ensure_columns" not in globals():
    def ensure_columns(df_source, feature_list):
        X_tmp = df_source[[c for c in feature_list if c in df_source.columns]].copy()
        missing_cols = [c for c in feature_list if c not in X_tmp.columns]
        for col in missing_cols:
            X_tmp[col] = 0.0
        return X_tmp[feature_list]

# Realign dataframes if processed data is available
if "df_used" in globals() and isinstance(df_used, pd.DataFrame) and not df_used.empty:
    X_full = ensure_columns(df_used, expected_x_full_features)
    X_selected = ensure_columns(df_used, expected_x_features)

    # Track mismatches for visibility
    missing_full = [c for c in expected_x_full_features if c not in df_used.columns]
    missing_x = [c for c in expected_x_features if c not in df_used.columns]

    print("✅ Feature lists synchronized.")
    print(f"  - X_features (expected {len(expected_x_features)}): {len(X_selected.columns)} columns")
    print(f"  - X_full_features (expected {len(expected_x_full_features)}): {len(X_full.columns)} columns")
    if missing_full or missing_x:
        print(f"⚠️ Missing in df_used -> X_full: {missing_full} | X: {missing_x}")
    else:
        print("🎯 All expected features are present in df_used.")
else:
    print("⚠️ df_used is not available; only feature lists were updated.")

✅ Feature lists synchronized.
  - X_features (expected 18): 18 columns
  - X_full_features (expected 28): 28 columns
🎯 All expected features are present in df_used.


In [30]:
# =============================================================================
# BUILD PREDICTIONS DATAFRAME (Close, Probability, True Target)
# USING ENSEMBLE OF TOP MODELS (MEAN AVERAGING)
# =============================================================================

import numpy as np
import pandas as pd

# Select top models by forward accuracy - include ALL models (no exclusions)
if 'MODELS_1m' not in globals() or not MODELS_1m:
    raise ValueError("MODELS_1m is empty. Please load 1m models first.")

# Include ALL models - no filtering
filtered_models = list(MODELS_1m.items())

# Get top models sorted by forward accuracy
top_3_models = sorted(filtered_models, key=lambda x: x[1]['forward_acc'], reverse=True)[:10]

print("🏆 TOP MODELS FOR ENSEMBLE:")
for i, (name, info) in enumerate(top_3_models, 1):
    print(f"  {i}. {name} - Forward Acc: {info['forward_acc']:.4f}, Feature Set: {info.get('feature_set', 'unknown')}")

# Resolve or build processed data
if 'DF_PROCESSED_1m' in globals() and isinstance(DF_PROCESSED_1m, pd.DataFrame) and not DF_PROCESSED_1m.empty:
    df_used = DF_PROCESSED_1m
elif 'df_processed' in globals() and isinstance(df_processed, pd.DataFrame) and not df_processed.empty:
    df_used = df_processed
elif 'df_proc' in globals() and isinstance(df_proc, pd.DataFrame) and not df_proc.empty:
    df_used = df_proc
else:
    # Build from already-loaded 1m raw data
    if 'DF_1M_MAX' in globals() and isinstance(DF_1M_MAX, pd.DataFrame) and not DF_1M_MAX.empty:
        df_used = calculate_indicators(DF_1M_MAX)
    elif 'DF_1M_MAX_EXTRA' in globals() and isinstance(DF_1M_MAX_EXTRA, pd.DataFrame) and not DF_1M_MAX_EXTRA.empty:
        df_used = calculate_indicators(DF_1M_MAX_EXTRA)
    else:
        raise ValueError("No processed 1m data found and no raw 1m data available. Run the 1m yfinance loader cell first.")

if df_used is None or df_used.empty:
    raise ValueError("Processed 1m data is empty. The dataset may be too short for indicators; load more data and rerun.")

# =============================================================================
# BUILD FEATURE MATRICES
# =============================================================================
# Get feature columns from models' configurations
features_config = None
for model_name, model_info in top_3_models:
    if model_info.get('features_config'):
        features_config = model_info['features_config']
        break

# Build full feature matrix from processed data
all_feature_cols = [c for c in df_used.columns if c not in ['target', 'return']]

# Use globally defined feature lists (X_FEATURES_1m and X_FULL_FEATURES_1m)
if 'X_FULL_FEATURES_1m' in globals() and X_FULL_FEATURES_1m:
    full_features = X_FULL_FEATURES_1m
    print(f"✅ Using X_FULL_FEATURES_1m: {len(full_features)} features")
elif features_config and 'full_features' in features_config:
    full_features = features_config['full_features']
    print(f"⚠️ Using features_config['full_features']: {len(full_features)} features")
else:
    full_features = all_feature_cols
    print(f"⚠️ Using all available columns: {len(full_features)} features")

if 'X_FEATURES_1m' in globals() and X_FEATURES_1m:
    selected_features = X_FEATURES_1m
    print(f"✅ Using X_FEATURES_1m: {len(selected_features)} features")
elif features_config and 'selected_features' in features_config:
    selected_features = features_config['selected_features']
    print(f"⚠️ Using features_config['selected_features']: {len(selected_features)} features")
else:
    selected_features = None
    print(f"⚠️ No selected features defined, will use full features")

# Helper to ensure all expected columns exist (pad missing with zeros, keep order)
def ensure_columns(df_source, feature_list):
    X_tmp = df_source[[c for c in feature_list if c in df_source.columns]].copy()
    missing_cols = [c for c in feature_list if c not in X_tmp.columns]
    for col in missing_cols:
        X_tmp[col] = 0.0
    return X_tmp[feature_list]

# Create feature sets - ensure columns exist
X_full = ensure_columns(df_used, full_features)
if selected_features:
    X_selected = ensure_columns(df_used, selected_features)
else:
    X_selected = X_full.copy()

print(f"\n📊 FEATURE SETS PREPARED:")
print(f"  - Full features (X_FULL_FEATURES_1m): {X_full.shape[1]} columns")
print(f"  - Selected features (X_FEATURES_1m): {X_selected.shape[1]} columns")
print(f"\n🔍 Feature Lists:")
print(f"  - X_FEATURES_1m: {list(X_selected.columns)}")
print(f"  - X_FULL_FEATURES_1m: {list(X_full.columns)}")

# =============================================================================
# ENSEMBLE PREDICTIONS - MEAN OF TOP MODELS
# Automatically detect required features from each model
# =============================================================================
all_probs = []
model_features_used = []
collected_model_names = []
min_len = len(df_used)

def _get_model_feature_count(model):
    """Robustly get expected feature count from a model, handling CatBoost and others."""
    # 1. Standard sklearn attribute
    if hasattr(model, 'n_features_in_') and model.n_features_in_ > 0:
        return model.n_features_in_
    # 2. CatBoost-specific: .feature_count_ or pool metadata
    if hasattr(model, 'feature_count_') and model.feature_count_ > 0:
        return model.feature_count_
    # 3. CatBoost method
    if callable(getattr(model, 'get_feature_count', None)):
        try:
            fc = model.get_feature_count()
            if fc and fc > 0:
                return fc
        except Exception:
            pass
    # 4. CatBoost feature_names_ length
    if hasattr(model, 'feature_names_') and model.feature_names_ is not None:
        names = model.feature_names_
        if isinstance(names, (list, tuple)) and len(names) > 0:
            return len(names)
    # 5. Fallback: generic n_features
    if hasattr(model, 'n_features') and model.n_features and model.n_features > 0:
        return model.n_features
    # 6. Keras: extract from input_shape (None, features) or (None, features, 1)
    input_shape = getattr(model, 'input_shape', None)
    if input_shape is not None:
        if len(input_shape) == 2 and input_shape[1] is not None and input_shape[1] > 0:
            return input_shape[1]
        if len(input_shape) == 3 and input_shape[1] is not None and input_shape[1] > 0:
            return input_shape[1]  # (batch, features, 1) -> features
    return None  # Unknown

for model_name, model_info in top_3_models:
    model = model_info['model']
    model_type = model_info.get('model_type', '')
    model_class = model_info.get('model_class', '') or ''
    feature_set_label = model_info.get('feature_set', 'X_selected')
    
    # Robustly detect expected number of features
    n_features_expected = _get_model_feature_count(model)
    
    # Check if this keras model needs 3D input (batch, features, 1) - for LSTM/GRU/RNN
    needs_3d_reshape = False
    input_shape = getattr(model, 'input_shape', None)
    if model_type == 'keras' and input_shape is not None and len(input_shape) == 3:
        needs_3d_reshape = True
    
    # Select appropriate feature matrix based on expected features
    if n_features_expected is not None:
        # Match by count: selected first (18), then full (28)
        if n_features_expected == X_selected.shape[1]:
            X_model = X_selected.copy()
            feature_type = f"selected ({X_selected.shape[1]} features)"
        elif n_features_expected == X_full.shape[1]:
            X_model = X_full.copy()
            feature_type = f"full ({X_full.shape[1]} features)"
        elif n_features_expected > max(X_selected.shape[1], X_full.shape[1]):
            # Model expects more features than any available set -> skip
            print(f"⏭️ Skipping {model_name}: expects {n_features_expected} features, but available are {X_selected.shape[1]} (selected) / {X_full.shape[1]} (full).")
            continue
        else:
            # Unknown count - default to selected (safer, smaller set)
            print(f"⚠️ {model_name}: Expected {n_features_expected} features, have {X_selected.shape[1]} (selected) or {X_full.shape[1]} (full). Using selected.")
            X_model = X_selected.copy()
            feature_type = f"selected ({X_selected.shape[1]} features, expected {n_features_expected})"
            # Pad or truncate if needed
            if X_model.shape[1] < n_features_expected:
                for i in range(n_features_expected - X_model.shape[1]):
                    X_model[f"_pad_{i}"] = 0.0
            elif X_model.shape[1] > n_features_expected:
                X_model = X_model.iloc[:, :n_features_expected]
    else:
        # No detectable feature count - use feature_set label with DEFAULT to selected
        if feature_set_label == 'X_full':
            X_model = X_full.copy()
            feature_type = f"full ({X_full.shape[1]} features, label: {feature_set_label})"
        else:
            # Default to selected for X, X_selected, or any unknown label
            X_model = X_selected.copy()
            feature_type = f"selected ({X_selected.shape[1]} features, label: {feature_set_label})"
    
    # Guard: skip models with zero columns after alignment (avoids errors)
    if X_model.shape[1] == 0:
        print(f"⏭️ Skipping {model_name}: feature matrix has 0 columns after alignment.")
        continue
    
    # Determine which scaler to use based on feature count
    if X_model.shape[1] == len(x_full_mean_dict_1m):
        mean_dict = x_full_mean_dict_1m
        std_dict = x_full_std_dict_1m
    else:
        mean_dict = x_mean_dict_1m
        std_dict = x_std_dict_1m

    # Z-score normalize using config values
    mean_s = pd.Series(mean_dict).reindex(X_model.columns).astype(float)
    std_s = pd.Series(std_dict).reindex(X_model.columns).astype(float).replace(0, np.nan)
    X_z = (X_model - mean_s) / std_s
    X_z = X_z.replace([np.inf, -np.inf], np.nan)
    X_z = X_z.fillna(0.0)
    
    # Reshape to 3D for LSTM/GRU/RNN keras models: (samples, features) -> (samples, features, 1)
    if needs_3d_reshape:
        X_input = X_z.values.reshape(X_z.shape[0], X_z.shape[1], 1)
        feature_type += " [3D reshaped]"
    else:
        X_input = X_z
    
    # Generate predictions for this model
    try:
        if hasattr(model, 'predict_proba'):
            proba = model.predict_proba(X_input)
            prob_up_model = proba[:, 1]
        else:
            # Fallback: use decision function or predictions
            if hasattr(model, 'decision_function'):
                scores = model.decision_function(X_input)
                prob_up_model = 1 / (1 + np.exp(-scores))
            else:
                preds = model.predict(X_input)
                preds = np.asarray(preds).reshape(-1)
                # For keras sigmoid output, values are already probabilities
                if model_type == 'keras':
                    prob_up_model = preds
                else:
                    prob_up_model = preds.astype(float)
        
        prob_up_model = np.asarray(prob_up_model).reshape(-1)
        min_len = min(min_len, len(prob_up_model))
        
        # Clip to valid range
        prob_up_model = np.clip(prob_up_model, 0, 1)
        all_probs.append(prob_up_model)
        model_features_used.append(feature_type)
        collected_model_names.append(model_name)
        
        print(f"✅ {model_name}: {feature_type} | mean prob: {prob_up_model.mean():.4f}")
        
    except Exception as e:
        print(f"❌ {model_name}: Prediction failed - {str(e)}")
        import traceback
        traceback.print_exc()
        continue

# Align lengths across models before averaging
if not all_probs:
    raise ValueError("No probabilities collected from models.")

if min_len <= 0:
    raise ValueError("Probabilities arrays are empty.")

all_probs_aligned = [probs[:min_len] for probs in all_probs]
prob_up_ensemble = np.mean(all_probs_aligned, axis=0)

print(f"\n📊 ENSEMBLE STATISTICS:")
print(f"  - Individual model probabilities: {len(all_probs_aligned)} models")
print(f"  - Ensemble mean probability: {prob_up_ensemble.mean():.4f}")
print(f"  - Ensemble std: {prob_up_ensemble.std():.4f}")
print(f"  - Ensemble range: [{prob_up_ensemble.min():.4f}, {prob_up_ensemble.max():.4f}]")

# Build dataframe with Close, Probability, True Target (aligned length)
preds_df = pd.DataFrame({
    'Close': df_used['Close'].values[:min_len],
    'Pred_Prob_Up': prob_up_ensemble,
    'True_Target': df_used['target'].values[:min_len]
}, index=df_used.index[:min_len])

# Store individual model predictions for analysis (only for collected models)
for i, model_name in enumerate(collected_model_names):
    preds_df[f'Prob_Model_{i+1}'] = all_probs_aligned[i]

print(f"\n✅ Predictions dataframe created using {len(collected_model_names)}-MODEL ENSEMBLE (mean averaging)")
print(preds_df.head())

# Store for later use
PREDICTIONS_1M_DF = preds_df.copy()
TOP_3_MODEL_NAMES = collected_model_names
MODEL_FEATURES_USED = model_features_used

print(f"\n🎯 Ensemble models:")
for name, features in zip(TOP_3_MODEL_NAMES, MODEL_FEATURES_USED):
    print(f"   - {name}: {features}")

🏆 TOP MODELS FOR ENSEMBLE:
  1. CatBoost_X - Forward Acc: 0.5226, Feature Set: X
  2. CatBoost_Full - Forward Acc: 0.5213, Feature Set: Full
  3. LightGBM_X - Forward Acc: 0.5174, Feature Set: X
  4. XGBoost_X - Forward Acc: 0.5157, Feature Set: X
  5. GRU_Full - Forward Acc: 0.5157, Feature Set: Full
  6. LSTM_X - Forward Acc: 0.5148, Feature Set: X
  7. LightGBM_Full - Forward Acc: 0.5143, Feature Set: Full
  8. AdaBoost_X - Forward Acc: 0.5142, Feature Set: X
  9. RandomForest_X - Forward Acc: 0.5137, Feature Set: X
  10. AdaBoost_Full - Forward Acc: 0.5137, Feature Set: Full
✅ Using X_FULL_FEATURES_1m: 28 features
✅ Using X_FEATURES_1m: 18 features

📊 FEATURE SETS PREPARED:
  - Full features (X_FULL_FEATURES_1m): 28 columns
  - Selected features (X_FEATURES_1m): 18 columns

🔍 Feature Lists:
  - X_FEATURES_1m: ['Open', 'Volume', 'RSI_20', 'ROC_10', 'CCI_20', 'MACD', 'MACD_hist', 'ATR_14', 'BW_20', 'CMF_20', 'Log_Return', 'Lag_1', 'Lag_2', 'Lag_3', 'Lag_4', 'Lag_5', 'Volume_lag1', 'V

In [31]:
# ============================================================================
# Summary Statistics for Predicted Probabilities
# ============================================================================

# Check if the predictions dataframe exists
if 'PREDICTIONS_1M_DF' in globals() and isinstance(PREDICTIONS_1M_DF, pd.DataFrame) and not PREDICTIONS_1M_DF.empty:
    # Extract the probabilities
    prob_up = PREDICTIONS_1M_DF['Pred_Prob_Up']
    
    # Calculate the range of probabilities
    prob_range = (prob_up.min(), prob_up.max())
    
    # Calculate the number of rows
    num_rows = prob_up.shape[0]
    
    # Calculate the mean and standard deviation
    prob_mean = prob_up.mean()
    prob_std = prob_up.std()
    
    # Display the summary statistics
    print("📊 Summary Statistics for Predicted Probabilities:")
    print(f"  - Range: {prob_range}")
    print(f"  - Number of Rows: {num_rows}")
    print(f"  - Mean Probability: {prob_mean:.4f}")
    print(f"  - Standard Deviation: {prob_std:.4f}")
else:
    print("❌ 'PREDICTIONS_1M_DF' is not available or is empty.")


📊 Summary Statistics for Predicted Probabilities:
  - Range: (np.float64(0.39065155739873975), np.float64(0.6190638986487171))
  - Number of Rows: 8410
  - Mean Probability: 0.4930
  - Standard Deviation: 0.0470


### sentiment analysis add to strategy 

### test strategy 

In [32]:
# ============================================================================
# Simple Buy/Neutral Strategy Based on Predicted Probabilities with Fixed Buy and Sell Thresholds
# ============================================================================
# - Buy Threshold:   0.6000
#   - Sell Threshold:  0.3240
#   - Stop Loss:       0.0100 (1.00%)
#   - Final Balance:   $1045.38
#   - Total Return:    +4.54%
#   - Trades:          8
# Define fixed thresholds for the probability to trigger buy and sell actions
stop_loss_pct = 1 / 100  # 3.5% stop loss
trading_cost_pct = 0.001  # 0.1% trading cost
buy_threshold = 0.6  # Fixed Buy threshold
sell_threshold = 0.324  # Fixed Sell threshold

# Initialize strategy variables
initial_balance = 1000.0
balance = initial_balance
units = 0  # Number of units owned - CRITICAL FIX
position = 0
entry_price = 0
trade_history = []

if 'PREDICTIONS_1M_DF' in globals() and isinstance(PREDICTIONS_1M_DF, pd.DataFrame) and not PREDICTIONS_1M_DF.empty:
    for _, row in PREDICTIONS_1M_DF.iterrows():
        price = row['Close']
        prob = row['Pred_Prob_Up']
        low_price = row.get('Low', price)  # Use Low if available

        if prob > buy_threshold and position == 0:
            entry_price = price * (1 + trading_cost_pct)
            units = balance / entry_price  # FIXED: Calculate units
            position = 1
            balance = 0  # FIXED: All cash is now invested
            trade_history.append({
                'action': 'BUY',
                'price': price,
                'entry_price': entry_price,
                'prob': prob,
                'units': units
            })

        elif position == 1:
            stop_loss_price = entry_price * (1 - stop_loss_pct)

            if low_price <= stop_loss_price:
                exit_price = stop_loss_price * (1 - trading_cost_pct)
                proceeds = units * exit_price  # FIXED: Multiply by units
                pct_return = (exit_price / entry_price) - 1
                balance = proceeds
                
                trade_history.append({
                    'action': 'STOP LOSS',
                    'price': price,
                    'exit_price': exit_price,
                    'entry_price': entry_price,
                    'prob': prob,
                    'return_pct': pct_return * 100,
                    'balance_after': balance
                })
                position = 0
                units = 0

            elif prob < sell_threshold:
                exit_price = price * (1 - trading_cost_pct)
                proceeds = units * exit_price  # FIXED: Multiply by units
                pct_return = (exit_price / entry_price) - 1
                balance = proceeds
                
                trade_history.append({
                    'action': 'SELL',
                    'price': price,
                    'exit_price': exit_price,
                    'entry_price': entry_price,
                    'prob': prob,
                    'return_pct': pct_return * 100,
                    'balance_after': balance
                })
                position = 0
                units = 0

    # Close any remaining open position at last price
    if position == 1:
        last_price = PREDICTIONS_1M_DF['Close'].iloc[-1]
        exit_price = last_price * (1 - trading_cost_pct)
        balance = units * exit_price  # FIXED
        pct_return = (exit_price / entry_price) - 1
        
        trade_history.append({
            'action': 'CLOSE (EOD)',
            'price': last_price,
            'exit_price': exit_price,
            'entry_price': entry_price,
            'prob': PREDICTIONS_1M_DF['Pred_Prob_Up'].iloc[-1],
            'return_pct': pct_return * 100,
            'balance_after': balance
        })

    # Final report
    final_balance = balance
    total_return = (final_balance / initial_balance - 1) * 100
    
    # Count trades by type
    buy_count = sum(1 for t in trade_history if t['action'] == 'BUY')
    sell_count = sum(1 for t in trade_history if t['action'] == 'SELL')
    stop_loss_count = sum(1 for t in trade_history if t['action'] == 'STOP LOSS')
    
    # Calculate win rate from exits
    exit_trades = [t for t in trade_history if 'return_pct' in t]
    winning_trades = [t for t in exit_trades if t['return_pct'] > 0]
    win_rate = (len(winning_trades) / len(exit_trades) * 100) if exit_trades else 0
    
    print(f"\n{'='*60}")
    print(f"✅ STRATEGY EXECUTION COMPLETE")
    print(f"{'='*60}")
    print(f"\n💰 FINAL RESULTS:")
    print(f"  - Initial Balance:  ${initial_balance:.2f}")
    print(f"  - Final Balance:    ${final_balance:.2f}")
    print(f"  - Total Return:     {total_return:+.2f}%")
    print(f"  - Profit/Loss:      ${final_balance - initial_balance:+.2f}")
    
    print(f"\n📊 TRADING STATISTICS:")
    print(f"  - Total Entries:    {buy_count}")
    print(f"  - Regular Exits:    {sell_count}")
    print(f"  - Stop Loss Exits:  {stop_loss_count}")
    print(f"  - Win Rate:         {win_rate:.2f}%")
    
    print(f"\n📋 TRADE HISTORY (Last 10 trades):")
    for i, trade in enumerate(trade_history[:], 1):
        if trade['action'] == 'BUY':
            print(f"  {i}. {trade['action']} @ ${trade['price']:.4f} (prob: {trade['prob']:.4f}) | Entry: ${trade['entry_price']:.4f}")
        else:
            print(f"  {i}. {trade['action']} @ ${trade['price']:.4f} (prob: {trade['prob']:.4f}) | Return: {trade['return_pct']:+.2f}% | Balance: ${trade['balance_after']:.2f}")
    
    # ============================================================================
    # PERFORMANCE METRICS - Sharpe Ratio and Maximum Drawdown
    # ============================================================================
    
    # Extract strategy returns (ignoring stop-loss)
    strategy_returns = [trade['return_pct'] / 100 for trade in trade_history if 'return_pct' in trade]

    # Convert to pandas series for easier calculations
    strategy_returns = pd.Series(strategy_returns)
    
    # Sharpe Ratio (Risk-Free Rate assumed to be 0)
    excess_return = strategy_returns.mean()  # Excess return = mean of strategy returns
    volatility = strategy_returns.std()  # Volatility (standard deviation of returns)
    sharpe_ratio = excess_return / volatility if volatility != 0 else 0
    
    # Maximum Drawdown (MDD)
    def max_drawdown(equity_curve):
        peak = equity_curve.expanding(min_periods=1).max()
        drawdown = (equity_curve - peak) / peak
        return drawdown.min() * 100
    
    # Calculate cumulative returns for MDD calculation
    equity_curve = (1 + strategy_returns).cumprod()
    mdd = max_drawdown(equity_curve)
    
    # Print performance metrics
    print(f"\n📊 PERFORMANCE METRICS:")
    print(f"  - Sharpe Ratio:     {sharpe_ratio:.2f}")
    print(f"  - Maximum Drawdown: {mdd:.2f}%")

else:
    print("❌ 'PREDICTIONS_1M_DF' is not available or is empty.")



✅ STRATEGY EXECUTION COMPLETE

💰 FINAL RESULTS:
  - Initial Balance:  $1000.00
  - Final Balance:    $1051.52
  - Total Return:     +5.15%
  - Profit/Loss:      $+51.52

📊 TRADING STATISTICS:
  - Total Entries:    8
  - Regular Exits:    0
  - Stop Loss Exits:  7
  - Win Rate:         12.50%

📋 TRADE HISTORY (Last 10 trades):
  1. BUY @ $1979.2167 (prob: 0.6038) | Entry: $1981.1959
  2. STOP LOSS @ $1954.8055 (prob: 0.6015) | Return: -1.10% | Balance: $989.01
  3. BUY @ $1951.2144 (prob: 0.6032) | Entry: $1953.1656
  4. STOP LOSS @ $1926.7258 (prob: 0.5985) | Return: -1.10% | Balance: $978.14
  5. BUY @ $1963.1151 (prob: 0.6190) | Entry: $1965.0782
  6. STOP LOSS @ $1945.2654 (prob: 0.5819) | Return: -1.10% | Balance: $967.39
  7. BUY @ $1935.8163 (prob: 0.6191) | Entry: $1937.7521
  8. STOP LOSS @ $1917.3575 (prob: 0.6042) | Return: -1.10% | Balance: $956.76
  9. BUY @ $1917.1825 (prob: 0.6009) | Entry: $1919.0997
  10. STOP LOSS @ $1886.3004 (prob: 0.5854) | Return: -1.10% | Balance

### optimize strategy 

In [33]:
# ============================================================================
# OPTIMIZE BUY/SELL THRESHOLDS AND STOP LOSS WITH GRID SEARCH
# ============================================================================
import numpy as np
import pandas as pd

if 'PREDICTIONS_1M_DF' not in globals() or PREDICTIONS_1M_DF.empty:
    raise ValueError("PREDICTIONS_1M_DF is missing. Run the predictions dataframe cell first.")

# Configuration
initial_balance = 1000.0
trading_cost = TRADING_COST_PCT if 'TRADING_COST_PCT' in globals() else 0.001
min_trades_required = 5

# Threshold and stop loss grids
buy_grid = np.round(np.linspace(0.50, 0.80, 16), 3)
sell_grid = np.round(np.linspace(0.20, 0.49, 15), 3)
stop_loss_grid = np.round(np.linspace(0.01, 0.05, 9), 4)  # 1% to 5% stop loss

print("📊 OPTIMIZATION PARAMETERS:")
print(f"  - Buy thresholds: {len(buy_grid)} values ({buy_grid[0]:.3f} to {buy_grid[-1]:.3f})")
print(f"  - Sell thresholds: {len(sell_grid)} values ({sell_grid[0]:.3f} to {sell_grid[-1]:.3f})")
print(f"  - Stop loss: {len(stop_loss_grid)} values ({stop_loss_grid[0]:.4f} to {stop_loss_grid[-1]:.4f})")
print(f"  - Total combinations: {len(buy_grid) * len(sell_grid) * len(stop_loss_grid)}")

results = []

for buy_th in buy_grid:
    for sell_th in sell_grid:
        if sell_th >= buy_th:
            continue
        
        for sl_pct in stop_loss_grid:
            balance = initial_balance
            position = 0
            entry_price = None
            trades = 0

            for _, row in PREDICTIONS_1M_DF.iterrows():
                prob = row['Pred_Prob_Up']
                price = row['Close']

                if position == 0 and prob >= buy_th:
                    # Buy
                    entry_price = price * (1 + trading_cost)
                    position = 1
                elif position == 1:
                    # Check stop loss first (priority 1)
                    stop_loss_price = entry_price * (1 - sl_pct)
                    if price <= stop_loss_price:
                        # Stop loss triggered
                        exit_price = price * (1 - trading_cost)
                        balance *= (exit_price / entry_price)
                        position = 0
                        trades += 1
                    # Check sell threshold (priority 2)
                    elif prob <= sell_th:
                        # Sell
                        exit_price = price * (1 - trading_cost)
                        balance *= (exit_price / entry_price)
                        position = 0
                        trades += 1

            # If still in position, close at last price
            if position == 1:
                last_price = PREDICTIONS_1M_DF['Close'].iloc[-1] * (1 - trading_cost)
                balance *= (last_price / entry_price)
                trades += 1

            # Require minimum trades before recording result
            if trades < min_trades_required:
                continue

            total_return = (balance / initial_balance) - 1
            results.append({
                'buy_threshold': float(buy_th),
                'sell_threshold': float(sell_th),
                'stop_loss_pct': float(sl_pct),
                'final_balance': float(balance),
                'total_return': float(total_return),
                'trades': trades
            })

results_df = pd.DataFrame(results).sort_values('total_return', ascending=False)

if results_df.empty:
    raise ValueError(f"No parameter sets generated at least {min_trades_required} trades. Expand grids or data.")

best_row = results_df.iloc[0]
OPTIMAL_BUY_THRESHOLD = best_row['buy_threshold']
OPTIMAL_SELL_THRESHOLD = best_row['sell_threshold']
OPTIMAL_STOP_LOSS = best_row['stop_loss_pct']

print("\n" + "="*70)
print("✅ OPTIMIZATION COMPLETE")
print("="*70)
print(f"\n🎯 OPTIMAL PARAMETERS (Maximizing Total Return with >= {min_trades_required} trades):")
print(f"  - Buy Threshold:   {OPTIMAL_BUY_THRESHOLD:.4f}")
print(f"  - Sell Threshold:  {OPTIMAL_SELL_THRESHOLD:.4f}")
print(f"  - Stop Loss:       {OPTIMAL_STOP_LOSS:.4f} ({OPTIMAL_STOP_LOSS*100:.2f}%)")
print(f"  - Final Balance:   ${best_row['final_balance']:.2f}")
print(f"  - Total Return:    {best_row['total_return']*100:+.2f}%")
print(f"  - Trades:          {int(best_row['trades'])}")

print(f"\n📊 TOP 10 PARAMETER COMBINATIONS:")
display(results_df[['buy_threshold', 'sell_threshold', 'stop_loss_pct', 'final_balance', 'total_return', 'trades']].head(10))

# Create visualization: Heatmaps for best stop loss configurations
print(f"\n📈 ANALYSIS BY STOP LOSS LEVEL:")
for sl in sorted(results_df['stop_loss_pct'].unique())[:3]:  # Show top 3 stop loss levels
    subset = results_df[results_df['stop_loss_pct'] == sl].nlargest(1, 'total_return')
    if not subset.empty:
        row = subset.iloc[0]
        print(f"  - SL {sl*100:.2f}%: Buy={row['buy_threshold']:.3f}, Sell={row['sell_threshold']:.3f}, Return={row['total_return']*100:+.2f}%, Trades={int(row['trades'])}")


📊 OPTIMIZATION PARAMETERS:
  - Buy thresholds: 16 values (0.500 to 0.800)
  - Sell thresholds: 15 values (0.200 to 0.490)
  - Stop loss: 9 values (0.0100 to 0.0500)
  - Total combinations: 2160

✅ OPTIMIZATION COMPLETE

🎯 OPTIMAL PARAMETERS (Maximizing Total Return with >= 5 trades):
  - Buy Threshold:   0.6000
  - Sell Threshold:  0.2410
  - Stop Loss:       0.0150 (1.50%)
  - Final Balance:   $1036.57
  - Total Return:    +3.66%
  - Trades:          6

📊 TOP 10 PARAMETER COMBINATIONS:


,buy_threshold,sell_threshold,stop_loss_pct,final_balance,total_return,trades
444,0.6,0.241,0.015,1036.568419,0.036568,6
448,0.6,0.262,0.015,1036.568419,0.036568,6
464,0.6,0.345,0.015,1036.568419,0.036568,6
460,0.6,0.324,0.015,1036.568419,0.036568,6
468,0.6,0.366,0.015,1036.568419,0.036568,6
472,0.6,0.386,0.015,1036.568419,0.036568,6
456,0.6,0.304,0.015,1036.568419,0.036568,6
452,0.6,0.283,0.015,1036.568419,0.036568,6
436,0.6,0.200,0.015,1036.568419,0.036568,6
440,0.6,0.221,0.015,1036.568419,0.036568,6



📈 ANALYSIS BY STOP LOSS LEVEL:
  - SL 1.00%: Buy=0.600, Sell=0.262, Return=+3.41%, Trades=8
  - SL 1.50%: Buy=0.600, Sell=0.241, Return=+3.66%, Trades=6
  - SL 2.00%: Buy=0.600, Sell=0.386, Return=+3.22%, Trades=5


In [21]:
# ============================================================================
# OPTIMIZE BUY/SELL THRESHOLDS ONLY (NO STOP LOSS)
# ============================================================================
import numpy as np
import pandas as pd

if 'PREDICTIONS_1M_DF' not in globals() or PREDICTIONS_1M_DF.empty:
    print("❌ PREDICTIONS_1M_DF is missing. Make sure previous cells were run.")
else:
    # Configuration
    initial_balance_opt = 1000.0
    trading_cost_opt = trading_cost_pct if 'trading_cost_pct' in globals() else 0.001
    min_trades_required = 5

    # Threshold grids
    # Smaller steps for more precision
    buy_grid_only = np.round(np.linspace(0.50, 0.85, 36), 3)
    sell_grid_only = np.round(np.linspace(0.15, 0.49, 35), 3)

    print("📊 THRESHOLD-ONLY OPTIMIZATION PARAMETERS:")
    print(f"  - Buy thresholds: {len(buy_grid_only)} values ({buy_grid_only[0]} to {buy_grid_only[-1]})")
    print(f"  - Sell thresholds: {len(sell_grid_only)} values ({sell_grid_only[0]} to {sell_grid_only[-1]})")
    print(f"  - Total combinations: {len(buy_grid_only) * len(sell_grid_only)}")

    results_no_sl = []

    # Prepare data for faster iteration (using numpy arrays)
    close_prices = PREDICTIONS_1M_DF['Close'].values
    probs = PREDICTIONS_1M_DF['Pred_Prob_Up'].values

    for buy_th in buy_grid_only:
        for sell_th in sell_grid_only:
            if sell_th >= buy_th:
                continue
            
            balance = initial_balance_opt
            position = 0
            entry_price = 0.0
            trades = 0

            for i in range(len(close_prices)):
                prob = probs[i]
                price = close_prices[i]

                if position == 0:
                    if prob >= buy_th:
                        # Buy
                        entry_price = price * (1 + trading_cost_opt)
                        position = 1
                else:
                    if prob <= sell_th:
                        # Sell
                        exit_price = price * (1 - trading_cost_opt)
                        balance *= (exit_price / entry_price)
                        position = 0
                        trades += 1

            # Close any remaining open position at last price
            if position == 1:
                last_price = close_prices[-1] * (1 - trading_cost_opt)
                balance *= (last_price / entry_price)
                trades += 1

            # Skip combos with too few trades
            if trades < min_trades_required:
                continue

            results_no_sl.append({
                'buy_threshold': float(buy_th),
                'sell_threshold': float(sell_th),
                'final_balance': float(balance),
                'total_return': float((balance / initial_balance_opt) - 1),
                'trades': trades
            })

    results_no_sl_df = pd.DataFrame(results_no_sl).sort_values('total_return', ascending=False)

    if results_no_sl_df.empty:
        raise ValueError(f"No combinations produced at least {min_trades_required} trades. Expand grids or data.")

    best_no_sl = results_no_sl_df.iloc[0]
    print("\n" + "="*70)
    print("✅ THRESHOLD OPTIMIZATION COMPLETE (NO STOP LOSS)")
    print("="*70)
    print(f"\n🎯 BEST PARAMETERS (>= {min_trades_required} trades):")
    print(f"  - Buy Threshold:   {best_no_sl['buy_threshold']:.4f}")
    print(f"  - Sell Threshold:  {best_no_sl['sell_threshold']:.4f}")
    print(f"  - Final Balance:   ${best_no_sl['final_balance']:.2f}")
    print(f"  - Total Return:    {best_no_sl['total_return']*100:+.2f}%")
    print(f"  - Trades:          {int(best_no_sl['trades'])}")

    print(f"\n📊 TOP 10 COMBINATIONS:")
    display(results_no_sl_df[['buy_threshold', 'sell_threshold', 'final_balance', 'total_return', 'trades']].head(10))


📊 THRESHOLD-ONLY OPTIMIZATION PARAMETERS:
  - Buy thresholds: 36 values (0.5 to 0.85)
  - Sell thresholds: 35 values (0.15 to 0.49)
  - Total combinations: 1260

✅ THRESHOLD OPTIMIZATION COMPLETE (NO STOP LOSS)

🎯 BEST PARAMETERS (>= 5 trades):
  - Buy Threshold:   0.6300
  - Sell Threshold:  0.4900
  - Final Balance:   $992.51
  - Total Return:    -0.75%
  - Trades:          7

📊 TOP 10 COMBINATIONS:


,buy_threshold,sell_threshold,final_balance,total_return,trades
153,0.63,0.49,992.507177,-0.007493,7
151,0.63,0.47,991.893037,-0.008107,7
150,0.63,0.46,991.384123,-0.008616,7
152,0.63,0.48,991.187977,-0.008812,7
33,0.53,0.39,990.803085,-0.009197,22
22,0.52,0.39,990.001207,-0.009999,22
148,0.63,0.44,985.822026,-0.014178,7
11,0.51,0.39,985.460832,-0.014539,23
132,0.62,0.39,984.952981,-0.015047,13
0,0.50,0.39,981.831000,-0.018169,24


In [22]:
# ============================================================================
# OPTIMIZE FIXED BUY/SELL THRESHOLDS AND ROLLING WINDOW (WITH STOP LOSS REMOVED)
# ============================================================================
import numpy as np
import pandas as pd

if 'PREDICTIONS_1M_DF' not in globals() or PREDICTIONS_1M_DF.empty:
    print("❌ PREDICTIONS_1M_DF is missing. Make sure previous cells were run.")
else:
    # Configuration
    initial_balance_opt = 1000.0
    trading_cost_opt = trading_cost_pct if 'trading_cost_pct' in globals() else 0.001
    min_trades_required = 5

    # Threshold and window size grids
    # Smaller steps for more precision
    buy_grid_only = np.round(np.linspace(0.50, 0.85, 36), 3)
    sell_grid_only = np.round(np.linspace(0.15, 0.49, 35), 3)
    window_size_grid = [7, 10, 15, 20, 25]  # Rolling window sizes to test

    print("📊 OPTIMIZATION PARAMETERS:")
    print(f"  - Buy thresholds: {len(buy_grid_only)} values ({buy_grid_only[0]} to {buy_grid_only[-1]})")
    print(f"  - Sell thresholds: {len(sell_grid_only)} values ({sell_grid_only[0]} to {sell_grid_only[-1]})")
    print(f"  - Window sizes: {window_size_grid}")
    print(f"  - Total combinations: {len(buy_grid_only) * len(sell_grid_only) * len(window_size_grid)}")

    results_opt = []

    # Prepare data for faster iteration (using numpy arrays)
    close_prices = PREDICTIONS_1M_DF['Close'].values
    probs = PREDICTIONS_1M_DF['Pred_Prob_Up'].values

    for buy_th in buy_grid_only:
        for sell_th in sell_grid_only:
            if sell_th >= buy_th:
                continue
            for window_size in window_size_grid:
                # Initialize strategy variables
                balance = initial_balance_opt
                position = 0
                entry_price = 0.0
                prob_window = []
                trades = 0

                for i in range(len(close_prices)):
                    prob = probs[i]
                    price = close_prices[i]

                    # Update the probability window for rolling mean calculation
                    prob_window.append(prob)
                    if len(prob_window) > window_size:
                        prob_window.pop(0)

                    # Calculate the rolling mean of probabilities
                    mean_prob = sum(prob_window) / len(prob_window)

                    # Use fixed buy/sell thresholds
                    fixed_buy_th = buy_th  # Use the fixed buy threshold
                    fixed_sell_th = sell_th  # Use the fixed sell threshold

                    if position == 0:
                        # If not holding a position, buy if the probability is above the fixed buy threshold
                        if prob >= fixed_buy_th:
                            entry_price = price * (1 + trading_cost_opt)
                            position = 1
                    else:
                        # If holding a position, sell if the probability falls below the fixed sell threshold
                        if prob <= fixed_sell_th:
                            exit_price = price * (1 - trading_cost_opt)
                            balance *= (exit_price / entry_price)
                            position = 0
                            trades += 1

                # Close any remaining open position at last price
                if position == 1:
                    last_price = close_prices[-1] * (1 - trading_cost_opt)
                    balance *= (last_price / entry_price)
                    trades += 1

                # Store results only if minimum trades reached
                if trades < min_trades_required:
                    continue

                results_opt.append({
                    'buy_threshold': float(buy_th),
                    'sell_threshold': float(sell_th),
                    'window_size': window_size,
                    'final_balance': float(balance),
                    'total_return': float((balance / initial_balance_opt) - 1),
                    'trades': trades
                })

    # Convert results to a DataFrame and sort by total return
    results_opt_df = pd.DataFrame(results_opt).sort_values('total_return', ascending=False)

    if results_opt_df.empty:
        raise ValueError(f"No rolling-window combinations produced at least {min_trades_required} trades. Expand grids or data.")

    best_opt = results_opt_df.iloc[0]
    print("\n" + "="*70)
    print("✅ OPTIMIZATION COMPLETE (WITH FIXED THRESHOLDS)")
    print("="*70)
    print(f"\n🎯 BEST PARAMETERS (>= {min_trades_required} trades):")
    print(f"  - Buy Threshold:   {best_opt['buy_threshold']:.4f}")
    print(f"  - Sell Threshold:  {best_opt['sell_threshold']:.4f}")
    print(f"  - Rolling Window:  {best_opt['window_size']}")
    print(f"  - Final Balance:   ${best_opt['final_balance']:.2f}")
    print(f"  - Total Return:    {best_opt['total_return']*100:+.2f}%")
    print(f"  - Trades:          {int(best_opt['trades'])}")

    print(f"\n📊 TOP 10 COMBINATIONS:")
    display(results_opt_df[['buy_threshold', 'sell_threshold', 'window_size', 'final_balance', 'total_return', 'trades']].head(10))


📊 OPTIMIZATION PARAMETERS:
  - Buy thresholds: 36 values (0.5 to 0.85)
  - Sell thresholds: 35 values (0.15 to 0.49)
  - Window sizes: [7, 10, 15, 20, 25]
  - Total combinations: 6300

✅ OPTIMIZATION COMPLETE (WITH FIXED THRESHOLDS)

🎯 BEST PARAMETERS (>= 5 trades):
  - Buy Threshold:   0.6300
  - Sell Threshold:  0.4900
  - Rolling Window:  25.0
  - Final Balance:   $992.51
  - Total Return:    -0.75%
  - Trades:          7

📊 TOP 10 COMBINATIONS:


,buy_threshold,sell_threshold,window_size,final_balance,total_return,trades
769,0.63,0.49,25,992.507177,-0.007493,7
768,0.63,0.49,20,992.507177,-0.007493,7
767,0.63,0.49,15,992.507177,-0.007493,7
766,0.63,0.49,10,992.507177,-0.007493,7
765,0.63,0.49,7,992.507177,-0.007493,7
759,0.63,0.47,25,991.893037,-0.008107,7
758,0.63,0.47,20,991.893037,-0.008107,7
757,0.63,0.47,15,991.893037,-0.008107,7
755,0.63,0.47,7,991.893037,-0.008107,7
756,0.63,0.47,10,991.893037,-0.008107,7
